## Preprocessing step for Joung2023 Panel_b.R

Extracts the four matrices needed by Panel_b.R from the new pipeline outputs
and saves them as .npy files under Fig1/Joung2023/Panel_b/:

- `Partial_MCC_A.npy`  — TF activity eigengenes (n_comm community + 1 global)  shape (n_cells, n_comm+1)
- `Partial_MCC_BX.npy` — Rotated latent B @ X_opt (after alignment)             shape (n_cells, n_comm+1)  → `heatmap_z` in Panel_b.R
- `Partial_MCC_B.npy`  — Raw AE embeddings Z (before alignment)                 shape (n_cells, n_latent)  → `heatmap_z_hat` in Panel_b.R
- `Partial_MCC_X.npy`  — Learned mixing matrix X_opt                            shape (n_latent, n_comm+1)

BX_opt is computed via `optimize_partial_mcc(A, Z)` — gradient descent maximising
the mean partial MCC between columns of A and B @ X. This matches the T-Mex score
produced by `run_partial_mcc_perm_experiments` in the AE notebook.

Run once before executing Joung2023_Panel_b.R.

In [6]:
import os
#import sys
import numpy as np
import pandas as pd
import scanpy as sc

#sys.path.insert(0, '../../../src')
from gcrl.grn.eigengenes import compute_eigengenes

In [7]:
# ── Control panel ──────────────────────────────────────────────
pct   = 50
gamma = '1p0'

data_dir = '../../../../data/real/Joung2023'
ae_dir   = f'../../../../results/real/Joung2023/ae_{pct}pct_gamma{gamma}'
grn_dir  = f'{data_dir}/GRN'
out_dir  = './Joung2023/Panel_b'   # save under Fig1/ alongside Panel_b.R outputs

os.makedirs(out_dir, exist_ok=True)

In [8]:
# ── Load processed adata and compute eigengenes ────────────────
adata = sc.read_h5ad(f'{data_dir}/Joung2023_processed_{pct}pct_gamma{gamma}.h5ad')
compute_eigengenes(adata, mode='all_cells',
                   reference_query='intervention == "unperturbed"')

# A — ALL eigengenes: n_comm community eigengenes + 1 global (last column)
comm_ids   = list(adata.uns['X_comm_eig_comm_ids'])
n_comm     = len(comm_ids)
global_idx = adata.uns['X_comm_eig_global_index']   # = n_comm (last column index)
A          = adata.obsm['X_comm_eig'].astype(np.float32)   # (n_cells, n_comm+1)
n_factors  = A.shape[1]
print(f'comm_ids: {comm_ids},  global col index: {global_idx}')
print(f'A (all eigengenes) shape: {A.shape}')

comm_ids: [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],  global col index: 6
A (all eigengenes) shape: (45072, 7)


In [9]:
# ── Latent embeddings and optimal alignment ────────────────────
from gcrl.alignment.partial_mcc import optimize_partial_mcc

Z = np.load(f'{ae_dir}/embeddings.npy').astype(np.float32)   # (n_cells, n_latent)
print(f'Z (raw embeddings) shape: {Z.shape}')

# optimize_partial_mcc finds X such that B @ X ≈ A (maximises partial MCC)
# Returns: score (float), X_opt (n_latent, n_factors), BX_opt (n_cells, n_factors)
score, X_opt, BX_opt = optimize_partial_mcc(A, Z, lr=5e-2, steps=500, seed=42, device=None)
print(f'T-Mex (optimised): {score:.4f}')

BX_opt = BX_opt.astype(np.float32)   # rotated latent B @ X_opt  (after alignment)
X_opt  = X_opt.astype(np.float32)    # mixing matrix X_opt
# Z stays as raw embeddings            (before alignment)

print(f'\nAll consistent: n_factors={n_factors}')
print(f'  A={A.shape}, Z pre-rot={Z.shape}, BX_opt post-rot={BX_opt.shape}, X_opt={X_opt.shape}')

Z (raw embeddings) shape: (45072, 7)
T-Mex (optimised): 0.3958

All consistent: n_factors=7
  A=(45072, 7), Z pre-rot=(45072, 7), BX_opt post-rot=(45072, 7), X_opt=(7, 7)


In [10]:
# ── Save ───────────────────────────────────────────────────────
np.save(f'{out_dir}/Partial_MCC_A.npy',  A)       # TF activity eigengenes
np.save(f'{out_dir}/Partial_MCC_BX.npy', BX_opt)  # rotated latent B @ X_opt  (heatmap_z in Panel_b.R)
np.save(f'{out_dir}/Partial_MCC_B.npy',  Z)        # raw AE embeddings          (heatmap_z_hat in Panel_b.R)
np.save(f'{out_dir}/Partial_MCC_X.npy',  X_opt)   # mixing matrix X_opt

print(f'\nSaved to {out_dir}/ (under Fig1/):')
print('  Partial_MCC_A.npy   — TF activity eigengenes,            shape', A.shape)
print('  Partial_MCC_BX.npy  — rotated latent B @ X_opt,          shape', BX_opt.shape)
print('  Partial_MCC_B.npy   — raw AE embeddings (pre-rotation),  shape', Z.shape)
print('  Partial_MCC_X.npy   — mixing matrix X_opt,               shape', X_opt.shape)
print(f'\nExpected T-Mex (from optimize_partial_mcc): {score:.4f}')


Saved to ./Joung2023/Panel_b/ (under Fig1/):
  Partial_MCC_A.npy   — TF activity eigengenes,            shape (45072, 7)
  Partial_MCC_BX.npy  — rotated latent B @ X_opt,          shape (45072, 7)
  Partial_MCC_B.npy   — raw AE embeddings (pre-rotation),  shape (45072, 7)
  Partial_MCC_X.npy   — mixing matrix X_opt,               shape (7, 7)

Expected T-Mex (from optimize_partial_mcc): 0.3958
